# S2 cointegration — leverage (vol-target policy)

Set **`VT_TARGET_ANN_VOL_STAR`** by hand after reviewing the §2 surface (same pattern as
`05_stop_bakeoff.ipynb` / H-013 freeze cells). §2 grids unlevered daily pre-VT book returns;
§4 writes your typed value into `s2_star_stack.json` via `backtest.star_stack_io`.

Arithmetic Sharpe is roughly invariant to constant leverage; CAGR, Calmar, drawdown, and CVaR are not. This is **not** a prop-firm pass-rate study.

**Run first.** This notebook does not fetch, score, or rebuild the book: it only reads the exported unlevered base parquet. Missing file → loud error naming the export notebook and path.

- Frozen `04_backtest/s2_coint/artifacts/s2_star_stack.json` (desk setpoint is `VT_TARGET_ANN_VOL_STAR`, not a separate leverage JSON).
- Prerequisite: `04_backtest/s2_coint/notebooks/01_star_tearsheet.ipynb` §20 exports:
  - `01_data/data_files/s2_coint/s2_period_returns_base.parquet` — pre-VT daily book `ret` (full sample so IS/OOS can be split)
  - `01_data/data_files/s2_coint/s2_period_returns.parquet` — net post-VT (later MC / prop-firm)
- After §4, re-run `01_star_tearsheet.ipynb` §20 if sealed net must reflect the new target.

Half-Kelly is betting about half the theoretically growth-optimal fraction so noisy estimates of edge do not blow the account; CAGR is the constant yearly rate that turns $1 into ending wealth; Calmar is that CAGR divided by the worst peak-to-trough loss; and CVaR is the average outcome in the worst tail (here 5%).

## 0. Imports & Config


In [11]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.research import DEFAULT_STAR_STACK
from backtest.star_stack_io import require_star
from risk.analytics.leverage.apply import s2_frozen_cfg
from risk.analytics.leverage.loaders import load_s2_period_returns_base
from risk.analytics.leverage.plots import surface_cagr_figure, surface_calmar_figure, surface_dd_figure
from risk.analytics.leverage.report import compute_leverage_surface, freeze_vt_target_ann_vol
from risk.analytics.monte_carlo.loaders import find_repo_root

ROOT = find_repo_root(ROOT)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("ROOT", ROOT)

SLEEVE = "s2"
BAR = "D"
PERIODS_PER_YEAR = 252
SIGMA_WINDOW = 60
VT_STAR = "s1_vt"
DEFAULT_IS_END = "2021-12-31"
DEFAULT_TARGETS = [0.06, 0.08, 0.10, 0.12, 0.15, 0.18]
DEFAULT_DD_CAP = 0.25
PICK = "calmar"  # objective for §2 suggestion only
STAR_STACK_PATH = DEFAULT_STAR_STACK

CFG = s2_frozen_cfg(
    target_ann_vol=0.10,
    periods_per_year=PERIODS_PER_YEAR,
    sigma_window=SIGMA_WINDOW,
)
print("s2 vt family s1_vt sigma_window", SIGMA_WINDOW)
print("STAR_STACK_PATH", STAR_STACK_PATH)


ROOT c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
s2 vt family s1_vt sigma_window 60
STAR_STACK_PATH C:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\04_backtest\s2_coint\artifacts\s2_star_stack.json


## 1. Data Loading


In [12]:
BASE = load_s2_period_returns_base(ROOT)
print("base bars", len(BASE), BASE.index.min().date(), BASE.index.max().date())
print(BASE.tail())


base bars 9228 1990-01-02 2026-08-24
date
2026-08-18    0.009611
2026-08-19   -0.005860
2026-08-20    0.001994
2026-08-21   -0.002775
2026-08-24    0.000000
Name: base, dtype: float64


## 2. Vol-target surface (research)

Frozen VT family (`VOL_STAR=s1_vt`); only `target_ann_vol` changes on the grid. IS and OOS are reported separately. Half-Kelly is a **ceiling** for the §2 suggestion, not your final pick.

In [13]:
PACK = {}

pack = compute_leverage_surface(
    BASE,
    CFG,
    targets=list(DEFAULT_TARGETS),
    is_end=DEFAULT_IS_END,
    periods_per_year=PERIODS_PER_YEAR,
    max_oos_dd=float(DEFAULT_DD_CAP),
    pick=PICK,
)
PACK.clear()
PACK.update(pack)
sug = pack["suggestion"]
print("Half-Kelly vol ceiling (not an objective)", pack["half_kelly_vol"])
if sug.get("target_ann_vol") is not None:
    print(
        f"Suggested VT_TARGET_ANN_VOL_STAR={sug['target_ann_vol']:.4g} "
        f"(OOS {PICK}={sug.get(f'oos_{PICK}'):.4f}, reason={sug.get('reason')})"
    )
else:
    print("No policy survivor — set VT_TARGET_ANN_VOL_STAR by judgment.")
print("Copy your choice into VT_TARGET_ANN_VOL_STAR in §0, then run §4.")
display(pack["surface"])
display(pd.Series(sug, name="suggestion").to_frame("value"))
display(surface_cagr_figure(pack["surface"]))
display(surface_calmar_figure(pack["surface"]))
display(surface_dd_figure(pack["surface"]))


Half-Kelly vol ceiling (not an objective) 0.6646385958633928
Suggested VT_TARGET_ANN_VOL_STAR=0.06 (OOS calmar=3.1387, reason=ok)
Copy your choice into VT_TARGET_ANN_VOL_STAR in §0, then run §4.


,target_ann_vol,realized_vol,is_sharpe,is_cagr,is_max_drawdown,is_calmar,is_cvar,oos_sharpe,oos_cagr,oos_max_drawdown,oos_calmar,oos_cvar,oos_n,is_n
0,0.06,0.061616,1.387574,0.083650,-0.075955,1.101308,-0.007799,1.560792,0.123536,-0.039359,3.138719,-0.007025,1164.0,8064.0
1,0.08,0.074449,1.395708,0.101933,-0.078609,1.296699,-0.009420,1.521942,0.147123,-0.052188,2.819107,-0.009197,1164.0,8064.0
2,0.10,0.084743,1.394858,0.115904,-0.078057,1.484872,-0.010623,1.486613,0.166257,-0.064873,2.562795,-0.011164,1164.0,8064.0
3,0.12,0.092974,1.390414,0.126461,-0.078093,1.619374,-0.011523,1.464643,0.183188,-0.078410,2.336279,-0.012830,1164.0,8064.0
4,0.15,0.102031,1.393065,0.139037,-0.079501,1.748862,-0.012467,1.400226,0.193780,-0.104053,1.862321,-0.014907,1164.0,8064.0
5,0.18,0.108463,1.391983,0.147927,-0.079501,1.860689,-0.013144,1.341498,0.196614,-0.124775,1.575755,-0.016409,1164.0,8064.0


,value
target_ann_vol,0.06
pick,calmar
n_survivors,6
half_kelly_vol,0.664639
max_oos_dd,0.25
oos_calmar,3.138719
oos_cagr,0.123536
oos_max_drawdown,-0.039359
oos_sharpe,1.560792
reason,ok


## 3. Suggested target (reference only)

Policy suggestion from §2 — **not** written to the star stack. Override in §0 if you disagree.

In [14]:
if not PACK:
    raise RuntimeError("run §2 first")
sug = PACK["suggestion"]
print("suggested target_ann_vol", sug.get("target_ann_vol"), "reason", sug.get("reason"))
print("survivors", sug.get("n_survivors"), "half-Kelly ceiling", PACK["half_kelly_vol"])
display(pd.Series(sug, name="suggestion").to_frame("value"))


suggested target_ann_vol 0.06 reason ok
survivors 6 half-Kelly ceiling 0.6646385958633928


,value
target_ann_vol,0.06
pick,calmar
n_survivors,6
half_kelly_vol,0.664639
max_oos_dd,0.25
oos_calmar,3.138719
oos_cagr,0.123536
oos_max_drawdown,-0.039359
oos_sharpe,1.560792
reason,ok


## 4. Type `VT_TARGET_ANN_VOL_STAR` then freeze

Confirm §0 matches your desk pick, then persist to `s2_star_stack.json` (merge via `update_star_stack_key`). Re-run `01_star_tearsheet.ipynb` §20 for sealed net if needed.

In [15]:
# Type STAR here
VT_TARGET_ANN_VOL_STAR = 0.06

require_star("VT_TARGET_ANN_VOL_STAR", VT_TARGET_ANN_VOL_STAR)
if VT_TARGET_ANN_VOL_STAR not in DEFAULT_TARGETS:
    raise ValueError(
        f"VT_TARGET_ANN_VOL_STAR={VT_TARGET_ANN_VOL_STAR} not in DEFAULT_TARGETS; "
        "add it to the grid or pick a listed value"
    )
if not PACK:
    raise RuntimeError("run §2 first")

row = PACK["surface"].loc[
    PACK["surface"]["target_ann_vol"].astype(float) == float(VT_TARGET_ANN_VOL_STAR)
]
if row.empty:
    raise ValueError("chosen target missing from §2 surface — re-run §2")
display(row.T)

hk = float(PACK["half_kelly_vol"])
if float(VT_TARGET_ANN_VOL_STAR) > hk + 1e-12:
    print(f"warning: {VT_TARGET_ANN_VOL_STAR:.3g} > half-Kelly ceiling {hk:.3g}")
dd_cap = float(PACK["max_oos_dd"])
oos_dd = float(row.iloc[0]["oos_max_drawdown"])
if oos_dd < -dd_cap:
    print(f"warning: OOS max DD {oos_dd:.3%} breaches veto {dd_cap:.0%}")

stack = freeze_vt_target_ann_vol(
    STAR_STACK_PATH,
    float(VT_TARGET_ANN_VOL_STAR),
    pick=PICK,
)
print(f"wrote {STAR_STACK_PATH}")
print(f"VT_TARGET_ANN_VOL_STAR={stack['VT_TARGET_ANN_VOL_STAR']}")


,0
target_ann_vol,0.060000
realized_vol,0.061616
is_sharpe,1.387574
is_cagr,0.083650
is_max_drawdown,-0.075955
is_calmar,1.101308
is_cvar,-0.007799
oos_sharpe,1.560792
oos_cagr,0.123536
oos_max_drawdown,-0.039359


wrote C:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\04_backtest\s2_coint\artifacts\s2_star_stack.json
VT_TARGET_ANN_VOL_STAR=0.06


## 5. Evaluation

- OOS Sharpe should be roughly flat across the vol grid (constant-$k$ invariance of arithmetic Sharpe).
- §2 suggestion is max OOS **Calmar** among targets that pass the drawdown veto and sit at or below half-Kelly vol; your §4 pick may differ.
- Label: live VT overlay on **base** returns (`s1_vt` family, only `target_ann_vol` changes).
- Next: `02_ev_vs_spy.ipynb` (geometry on sealed net) and `prop_firm/` (FTMO-capped $k$).

In [16]:
print("sleeve", SLEEVE, "desk target", VT_TARGET_ANN_VOL_STAR)
print("half-Kelly is a ceiling, not an objective")
if PACK.get("suggestion"):
    print("§2 suggestion", PACK["suggestion"])


sleeve s2 desk target 0.06
half-Kelly is a ceiling, not an objective
§2 suggestion {'target_ann_vol': 0.06, 'pick': 'calmar', 'n_survivors': 6, 'half_kelly_vol': 0.6646385958633928, 'max_oos_dd': 0.25, 'oos_calmar': 3.1387191247165136, 'oos_cagr': 0.12353622087368277, 'oos_max_drawdown': -0.039358800824473406, 'oos_sharpe': 1.5607919768030292, 'reason': 'ok'}
